# Test Attendance Record Notifications

This notebook tests Socket.IO notifications when creating attendance records.

**⚠️ IMPORTANT: Login as ORGANIZATION ADMIN, not superadmin!**
- Superadmin won't get notifications for org-specific records
- Use the same org admin credentials in both notebook and frontend

**Before running:**
1. Open frontend in browser and login as **ORG ADMIN**
2. Open browser console (F12)
3. Watch the bell icon for notifications

**Expected results:**
- Browser console: `✅ Record added notification`
- Toast notification: "{Username} is now in/out"
- Bell icon badge count increases

## 📦 Install Dependencies (Run Once if Needed)

In [ ]:
# Only run this if you get import errors
import sys
!{sys.executable} -m pip install requests python-dotenv --quiet

print('✅ Dependencies installed!')

## 🚀 Setup (RUN THIS CELL FIRST!)

In [ ]:
# ⚠️ RUN THIS CELL FIRST - Setup imports and configuration
import requests
import os
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()

# Configuration - Use ORG ADMIN credentials (NOT superadmin!)
# Superadmin won't get notifications for org-specific attendance records
BACKEND_URL = os.getenv('SO_BACKEND_API_URL', 'http://localhost:7091')

session = requests.Session()
session.headers.update({"Content-Type": "application/json", "accept": "application/json"})

print('🎉 Setup complete!')
print(f'✅ Backend API: {BACKEND_URL}')
print('\n✅ All imports loaded: requests, os, datetime')
print('\n⚠️  IMPORTANT: Login to frontend with SAME org admin credentials!')
print('   Notifications only work if you\'re logged in as the org admin')
print('\nYou can now run the cells below!')

## 🔐 Login

In [ ]:
# Login to backend as ORG ADMIN
def login_to_backend(email=None, password=None, client_slug='humblebee'):
    """
    Login to backend and return auth token

    Args:
        email: Admin email (ORG ADMIN, NOT superadmin!)
        password: Admin password
        client_slug: Organization slug (default: 'humblebee')
    """
    # Use provided credentials or fall back to env vars
    if email is None:
        email = os.getenv('SO_ADMIN_EMAIL', 'admin@humblebee.ai')
    if password is None:
        password = os.getenv('SO_ADMIN_PASSWORD', 'admin123')

    print(f'🔐 Logging in as {email} to org "{client_slug}"...')

    # Login payload - backend expects: { email, password, client_slug }
    response = session.post(
        f'{BACKEND_URL}/api/auth/login',
        json={
            'email': email,
            'password': password,
            'client_slug': client_slug
        }
    )

    if response.status_code == 200:
        data = response.json()
        token = data.get('token') or data.get('accessToken')
        session.headers.update({'Authorization': f'Bearer {token}'})
        print(f'✅ Login successful')
        print(f'   User: {data.get("user", {}).get("full_name")} ({data.get("user", {}).get("email")})')
        print(f'   Role: {data.get("user", {}).get("role")}')
        print(f'   Organization: {client_slug}')
        return token, client_slug
    else:
        print(f'❌ Login failed: {response.status_code}')
        print(response.text)
        return None, None

# 🎯 EDIT THESE CREDENTIALS (Use your ORG ADMIN credentials):
auth_token, slug = login_to_backend(
    email='humblebee@gmail.com',      # Change to your org admin email
    password='Humblebee2025@',         # Change to your org admin password
    client_slug='humblebee'            # Change to your org slug
)

## 👥 Get Users

In [ ]:
# Use the organization slug from login
print(f"Using organization: {slug}")

In [ ]:
# Get list of users
def list_org_users(slug, page=1, limit=50):
    """Get users from organization"""
    r = session.get(f"{BACKEND_URL}/api/org/{slug}/users", params={"page": page, "limit": limit})
    r.raise_for_status()
    return r.json()

users = list_org_users(slug)
print(f"Found {len(users)} users in '{slug}'")
print("\nFirst 5 users:")
for u in users[:5]:
    print(f"  ID: {u.get('id')}, Name: {u.get('full_name')}, Email: {u.get('email')}")

## 📝 Create Attendance Record (EDIT USER_ID AND STATUS HERE!)

**⚠️ BEFORE RUNNING THIS CELL:**
1. Make sure frontend is open in browser
2. Browser console is open (F12)
3. Watch the bell icon

**EDIT THE VALUES BELOW:**
- `user_id`: Change to any user ID from the list above
- `status`: Change to "in" or "out"
- `camera_id`: Leave as None (no cameras configured) or set to valid camera ID

In [ ]:
# 🎯 EDIT THESE VALUES:
user_id = 61        # Change this to any user ID from the list above
status = "in"       # Change to "in" or "out"
camera_id = None    # Set to None (no camera) or a valid camera ID if you have cameras

# Helper function to create timestamp
def iso_now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

# Create the attendance record
payload = {
    "user_id": int(user_id),
    "status": status.lower(),
    "timestamp": iso_now()
}

# Only add camera_id if it's not None
if camera_id is not None:
    payload["camera_id"] = int(camera_id)

print(f'📝 Creating attendance record:')
print(f'   User ID: {user_id}')
print(f'   Status: {status.upper()}')
print(f'   Camera ID: {camera_id if camera_id else "None (manual entry)"}')
print(f'   Time: {payload["timestamp"]}')
print('\n' + '='*70)

r = session.post(f"{BACKEND_URL}/api/org/{slug}/attendance-records", json=payload)

# Check for errors and show details
if r.status_code != 201:
    print(f'\n❌ Error {r.status_code}: {r.reason}')
    print(f'Response body: {r.text[:500]}')
    print('='*70)
else:
    result = r.json()
    print('\n✅ Attendance record created successfully!')
    print(f'   Record ID: {result.get("id")}')
    print(f'   External ID: {result.get("external_id")}')
    print(f'   Status: {result.get("status")}')
    print(f'   Timestamp: {result.get("timestamp")}')
    print('\n' + '='*70)
    print('\n🔔 CHECK YOUR BROWSER NOW!')
    print('   Expected in console: "✅ Record added notification"')
    print('   Expected toast: "{Username} is now in/out"')
    print('   Expected: Bell icon badge count increases')
    print('='*70)

## 🔄 Quick Test: Create Multiple Records

This will create 3 attendance records quickly to test multiple notifications.

In [ ]:
import time

# Test with first 3 users, alternating IN/OUT
test_count = min(3, len(users))
statuses = ['in', 'out', 'in']

print(f'🚀 Creating {test_count} attendance records')
print('   Watch your browser for notifications!\n')
print('='*70)

for i in range(test_count):
    user = users[i]
    status = statuses[i % len(statuses)]

    payload = {
        "user_id": int(user['id']),
        "status": status,
        "timestamp": iso_now()
    }
    # Note: camera_id omitted (will be null) since we have no cameras configured

    print(f'\n{i+1}. Creating record: {user["full_name"]} -> {status.upper()}')

    try:
        r = session.post(f"{BACKEND_URL}/api/org/{slug}/attendance-records", json=payload)
        if r.status_code == 201:
            result = r.json()
            print(f'   ✅ Record ID: {result.get("id")}')
        else:
            print(f'   ❌ Failed: {r.status_code} - {r.text[:100]}')
    except Exception as e:
        print(f'   ❌ Failed: {e}')

    # Wait 1.5 seconds between records
    if i < test_count - 1:
        time.sleep(1.5)

print('\n' + '='*70)
print(f'✅ Created {test_count} records')
print('\n🔔 Check your browser:')
print(f'   - Should see {test_count} toast notifications')
print(f'   - Console should show {test_count} "✅ Record added notification"')
print(f'   - Bell icon badge should show count')
print('='*70)

## 🔍 Troubleshooting

If notifications don't appear, check:

In [ ]:
print("🔍 Troubleshooting Checklist:\n")
print("0. ⚠️  MOST IMPORTANT:")
print("   ✓ You MUST be logged in as ORG ADMIN in frontend")
print("   ✓ NOT superadmin - superadmin won't get org notifications")
print("   ✓ Use same credentials in notebook (cell 6) and frontend\n")

print("1. Frontend Connection:")
print("   ✓ Browser console should show: '🔌 Socket connected successfully'")
print("   ✓ Sidebar should show green 'Connected' status")
print("   ✓ Look for: '✅ Successfully joined client room'\n")

print("2. Record Creation:")
print("   ✓ Cell above shows '✅ Record created successfully'")
print("   ✓ Frontend shows 'Record saved successfully' toast\n")

print("3. If notifications still don't work:")
print("   ⚠️  Backend is NOT emitting socket events (most likely cause)")
print("   ⚠️  Check backend logs: docker compose logs backend | tail -50")
print("   ⚠️  Look for socket.emit('data_update', ...) in backend code")
print(f"   ⚠️  Verify backend emits to room: '{slug}'\n")

print("4. Backend should emit after creating record:")
print("   io.to(clientSlug).emit('data_update', {")
print("     action: 'record_added',")
print("     data: { username: 'User Name', type: 'in' }")
print("   })\n")

print(f"📊 Configuration:")
print(f"   API: {BACKEND_URL}")
print(f"   Organization: {slug}")
print(f"   Admin Email: {session.headers.get('Authorization', 'Not logged in')[:20]}...")
print(f"   Users loaded: {len(users) if 'users' in dir() else 0}")
print(f"   Authenticated: {'✅' if auth_token else '❌'}")